# DocStream — Ollama on Colab via ngrok

This notebook installs Ollama, pulls `llama3.1:8b`, and exposes it over the
internet via an ngrok tunnel so DocStream can use it as its AI fallback.

**Steps:**
1. Install dependencies
2. Start Ollama server
3. Pull the model
4. Open ngrok tunnel
5. Copy the tunnel URL into your `.env` as `OLLAMA_BASE_URL`

> **Tip:** Use a GPU runtime (Runtime → Change runtime type → T4 GPU) for faster inference.

In [ ]:
# ── 1. Install Ollama ──────────────────────────────────────────────────────────
!curl -fsSL https://ollama.com/install.sh | sh
!pip install pyngrok --quiet

In [ ]:
# ── 2. Start Ollama server in the background ───────────────────────────────────
import subprocess, time

proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)  # give the server a moment to start
print("Ollama server started (PID", proc.pid, ")")

In [ ]:
# ── 3. Pull the model (takes ~5 min on first run) ──────────────────────────────
!ollama pull llama3.1:8b

In [ ]:
# ── 4. Open ngrok tunnel ───────────────────────────────────────────────────────
# Set your ngrok auth token (free at https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTH_TOKEN = ""  # ← paste your token here

from pyngrok import ngrok, conf

if NGROK_AUTH_TOKEN:
    conf.get_default().auth_token = NGROK_AUTH_TOKEN

tunnel = ngrok.connect(11434, "http")
public_url = tunnel.public_url

print("\n" + "=" * 60)
print("Ollama public URL:")
print(f"  {public_url}")
print("=" * 60)
print("\nAdd this to your .env files:")
print(f"  OLLAMA_BASE_URL={public_url}/")
print("\nKeep this notebook running to maintain the tunnel.")

In [ ]:
# ── 5. Verify (optional) ───────────────────────────────────────────────────────
import requests

resp = requests.get(
    f"{public_url}/api/tags",
    headers={"ngrok-skip-browser-warning": "true"},
    timeout=10,
)
models = [m["name"] for m in resp.json().get("models", [])]
print("Available models:", models)